# Multi-label klasifikacija torakalnih bolesti

Ovaj projekat se bavi razvojem modela za automatsku detekciju patoloških stanja na osnovu rendgenskih snimaka grudnog koša. Problem je definisan kao multi-label klasifikacija, što znači da jedan snimak može istovremeno sadržavati nula, jednu ili više različitih dijagnoza. 

Fokus je na 14 specifičnih patologija:
Atelectasis, Cardiomegaly, Edema, Effusion, Emphysema, Fibrosis, Hernia, Infiltration, Mass, Nodule, Pleural Thickening, Pneumonia, Pneumothorax i Consolidation.

Rendgenski snimak grudnog koša je jedna od najčešćih dijagnostičkih metoda u medicini. Interpretacija ovih snimaka je lako podložna greškama uslijed različitih faktora. Automatizacija ovog procesa pomoću dubokog učenja nudi bržu identifikacija kritičnih stanja i analizu zasnovanu na hiljadama prethodnih primjera.

U izgradnji modela se koristi **NIH Chest X-ray** dataset, koji sadrži preko 112,000 snimaka toraksa, od preko 30 000 pacijenata.

Za izgradnju modela korišćena je **DenseNet-121** arhitektura.

## DenseNet-121

**DenseNet-121** je vrsta duboke neuronske mreže kod koje je svaki sloj povezan sa svim prethodnim slojevima unutar bloka. 

Za razliku od klasičnih CNN-ova, ovde svaki sloj dobija originalni ulaz, izlaz prvog sloja, izlaz drugog sloa, ..., i tako sve do sloja koji mu prethodi. To znači da informacije stalno cirkulišu kroz mrežu i ponovo se koriste. 

Takva struktura omogućava:
- bolji protok gradijenta (smanjuje problem nestajanja gradijenta tokom treninga) 
- ponovnu upotrebu feature-a (karakteristike koje su već izvučene ne moraju ponovo da se uče). 

Ova arhitektura izabrana je upravo zbog te mogućnosti ponovne upotrebe relevantnih karakteristika, stabilnog protoka gradijenta, ali i zbog činjenice da se pokazao efikasnim upravo za dijagnostifikovanje bolesti na osnovu medicinskih snimaka.

## Izrada modela:

Na početku učitavamo neophodne alate za rad sa podacima, slikama i modelom. 

Pandas i NmuPy za manipulaciju podacima.

TensorFlow i Keras za definisanje, treniranje i evaluaciju modela dubokog učenja.

Scikit-learn za dijeljenje podataka.

ImageDataGenerator za data augmentation i efikasno učitavanje slika sa diska u memoriju GPU tokom treninga.

DenseNet121 arhitektura koju koristimo kao osnovu modela.

Callbacks mehanizmi koji prate proces treniranja.

In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator 
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

Prvi konkretan korak jeste priprema podataka.

Prvo se definiše putanja do csv fajla koji sadži metapodatke o slikama. Kolona Finding Labels sadrži dijagnoze u tekstualnom obliku, pri čemu jedna slika može imati više bolesti odjednom. 

Kako bi se podaci prilagodili neuronskoj mreži, prvo se izdvajaju sve jedinstvene bolesti iz ove kolone. Nakon toga se uklanja oznaka No Finding jer ona ne predstavlja patologiju, već odsustvo iste. 

Zatim se vrši transformacija tekstualnih oznaka u numerički format primjenom takozvanog multi-hot encoding-a. Za svaku bolest se kreira posebna binarna kolona u DataFrame-u. Ukoliko je određena bolest prisutna na slici, vrijednost u toj koloni dobija vrijednost 1, dok u suprotnom iznosi 0. 

Nakon obrade oznaka, sledeći korak podrazumijeva povezivanje metapodataka sa stvarnim slikama. 

Na kraju, kompletan skup podataka se dijeli na trening, validacioni i test skup. Trening skup čini 70% podataka i koristi se za učenje modela, dok se preostalih 30% dijeli ravnomijerno na validacioni i test skup. 

Ovim postupkom su podaci transformisani u numerički i organizovan oblik pogodan za treniranje duboke neuronske mreže.

In [ ]:
base_path = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
csv_path = base_path + "/Data_Entry_2017.csv"

df = pd.read_csv(csv_path)

all_labels = (
    df["Finding Labels"]
    .str.split("|")
    .explode()
    .unique()
)

all_labels = sorted([label for label in all_labels if label != "No Finding"])  

for label in all_labels:
    df[label] = df["Finding Labels"].apply(lambda x: 1 if label in x else 0)

image_paths = {}
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".png"):
            image_paths[file] = os.path.join(root, file)

df["path"] = df["Image Index"].map(image_paths)
df = df.dropna(subset=["path"])

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

Nakon pripreme i podjele podataka, sledeći korak u izgradnji modela odnosi se na rješavanje problema neravnoteže klasa (class imbalance). U NIH Chest X-ray skupu podataka pojedine bolesti su znatno rjeđe zastupljene u odnosu na druge. Ukoliko bi se model trenirao bez dodatnih korekcija, imao bi tendenciju da favorizuje češće bolesti, dok bi performanse na rijetkim patologijama bile znatno slabije.

Kako bi se ovaj problem ublažio, izračunavaju se težine za svaku klasu na osnovu odnosa broja negativnih i pozitivnih uzoraka u trening skupu. Za svaku bolest određuje se broj pozitivnih primjera (slika na kojima je bolest prisutna) i broj negativnih primjera (slika bez te bolesti). Težina klase se zatim definiše kao odnos broja negativnih i pozitivnih uzoraka. Na taj način rjeđe bolesti dobijaju veću težinu, dok češće bolesti dobijaju manju težinu. Dodavanje male konstante (1e-5) u imenilac sprječava dijeljenje nulom u slučaju ekstremno rijetkih klasa.

Izračunate težine se zatim konvertuju u TensorFlow tenzor kako bi mogle biti korišćene tokom optimizacije modela.

Na osnovu ovih težina se definiše prilagođena funkcija gubitka - weighted binary crossentropy. Budući da je riječ o multi-label problemu, koristi se binarna entropija za svaku klasu pojedinačno, umjesto softmax funkcije. Ako bi neka klasa imala znatno više primjera od druge, model bi mogao da ignoriše tu klasu sa manje primjera i svejedno imao dobar ukupni rezultat. Ali korišćenjem ove funkcije model je primoran da obrati pažnju na rijetke patologije. Dakle ako model pogriješi kod rijetke bolesti dobija veću kaznu. 

$$BCE = -[y \log(p) + (1 - y) \log(1 - p)]$$

$$WBCE = -[w \cdot y \log(p) + (1 - y) \log(1 - p)]$$

| Simbol | Naziv | Opis |
| :---: | :--- | :--- |
| $y$ | **Stvarna vrijednost** | Labela iz skupa podataka (0 ili 1). |
| $p$ | **Predikcija modela** | Vjerovatnoća koju model predviđa. |
| $w$ | **Težinski faktor** | Izračunata težina: $\frac{\text{broj negativnih}}{\text{broj pozitivnih}}$. |

In [ ]:
pos_weights = []
for label in all_labels:
    pos = train_df[label].sum()
    neg = len(train_df) - pos
    pos_weights.append(neg / (pos + 1e-5))

pos_weights_tensor = tf.constant(np.array(pos_weights), dtype=tf.float32)

def weighted_binary_crossentropy(y_true, y_pred):
    epsilon = 1e-7
    y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
    loss = - (pos_weights_tensor * y_true * tf.math.log(y_pred) +
              (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

Sledeći korak u izgradnji modela odnosi se na pripremu slika za treniranje neuronske mreže. Budući da se koristi DenseNet-121 arhitektura u okviru transfer learning pristupa, ulazne slike je potrebno prilagoditi standardnoj dimenziji koju model očekuje. U tom smislu definiše se veličina slike od 224×224 piksela, što je standardna ulazna rezolucija za modele trenirane na ImageNet datasetu. Takođe se definiše veličina batch-a od 32 slike, što predstavlja broj uzoraka koji se istovremeno obrađuju tokom jednog koraka optimizacije.

Kako bi se poboljšala generalizacija modela i smanjio rizik od overfitting-a, primjenjuje se data augmentation nad trening skupom. U tu svrhu koristi se ImageDataGenerator, koji omogućava dinamičku transformaciju slika tokom treninga. Sve slike se prvo skaliraju dijeljenjem vrijednosti piksela sa 255, čime se normalizuju u opseg [0,1]. Pored toga, primjenjuju se blage geometrijske transformacije: rotacija do 10 stepeni, horizontalno i vertikalno pomijeranje do 5%, zumiranje do 10%, kao i horizontalno preslikavanje (horizontal flip). Ove transformacije simuliraju različite varijacije u položaju i orijentaciji pacijenta, čime se model čini robusnijim na male promjene u ulaznim podacima.

Važno je naglasiti da se augmentacija primjenjuje isključivo nad trening skupom. Validacioni skup prolazi samo kroz proces skaliranja, bez dodatnih transformacija, kako bi evaluacija performansi modela bila realistična i odražavala stvarne podatke.

Funkcija flow_from_dataframe čita slike sa diska, prilagođava ih, uzima njihove labele iz DataFrame-a i vraća ih u batch-evima. Znači, umjesto da učitamo sve slike u RAM, generator ih učitava postepeno, tokom treninga.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="path",
    y_col=all_labels,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="raw"
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="path",
    y_col=all_labels,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="raw"
)